# Data Collection

## 1. Bitcoin Historical Price Data

In [2]:
%run 00_functions.ipynb

c:\Users\nyy32\OneDrive\Documents\UM_Master of Artifcial Inteligence\Thesis\Github\Cryptocurrency Forecasting\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
latest_to_date = datetime.now().strftime("%m/%d/%Y")

original_investing_info = historical_data.__globals__["investing_info"]

def investing_info_with_volume(investing_id):
    info = original_investing_info(investing_id)
    info["has_no_volume"] = False
    return info

historical_data.__globals__["investing_info"] = investing_info_with_volume

try:
    data = historical_data(
        investing_id=1057391,
        from_date="01/01/2018",
        to_date=latest_to_date,
        interval="D"
    )
finally:
    historical_data.__globals__["investing_info"] = original_investing_info

print("Available columns:", data.keys())
print("Number of dates:", len(data["date"]))
print("Number of volume values:", len(data["volume"]))
print("Latest date:", data["date"][-1])

INFO: HTTP Request: GET https://tvc6.investing.com/0ede04bc9a3d410b8342c47843d26699/0/0/0/0/symbols?symbol=1057391 "HTTP/1.1 403 Forbidden"


ConnectionError: Request to Investing.com API failed with error code: 403.

In [ ]:
data_df = pd.DataFrame(data)

data_df.head()

,date,open,high,low,close,volume
0,01/01/2018,13850.490234,13921.530273,12877.669922,13444.879883,78425
1,01/02/2018,13444.879883,15306.129883,12934.160156,14754.129883,137732
2,01/03/2018,14754.089844,15435.009766,14579.709961,15156.620117,106543
3,01/04/2018,15156.490234,15408.660156,14244.669922,15180.080078,110969
4,01/05/2018,15180.080078,17126.949219,14832.360352,16954.779297,141960


In [ ]:
data_df.to_csv("../data/lstm/raw/bitcoin_historical_data.csv", index=False)

# 2. News Data Collection

## Objective
Collect recent and current Bitcoin-related financial news
from Yahoo Finance (BTC-USD).

## Required Data
- Publication date
- Headline
- News source
- Article content or available summary

In [ ]:
# Configuration
NEWS_URL = "https://finance.yahoo.com/quote/BTC-USD/news/"
MAX_SCROLL_ROUNDS = 30
WAIT_AFTER_SCROLL_MS = 2500

# Reuse one session for article-page requests
session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0.0.0 Safari/537.36"
    )
})


def parse_timestamp(timestamp):
    """Parse an ISO timestamp and normalize it to UTC."""
    if not timestamp:
        return None

    try:
        parsed = datetime.fromisoformat(
            timestamp.replace("Z", "+00:00")
        )
        if parsed.tzinfo is None:
            return None
        return parsed.astimezone(timezone.utc)
    except (ValueError, TypeError):
        return None


def get_published_time(article_url):
    """Retrieve the exact publication timestamp from article metadata."""
    response = session.get(article_url, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    def find_timestamp(data):
        items = data if isinstance(data, list) else [data]

        for item in items:
            if not isinstance(item, dict):
                continue

            if item.get("datePublished"):
                return item["datePublished"]

            nested_items = item.get("@graph", [])
            if nested_items:
                timestamp = find_timestamp(nested_items)
                if timestamp:
                    return timestamp

        return None

    for script in soup.find_all(
        "script",
        type="application/ld+json"
    ):
        try:
            data = json.loads(script.string or script.get_text())
        except (json.JSONDecodeError, TypeError):
            continue

        timestamp = find_timestamp(data)
        if timestamp:
            return timestamp

    return None


async def collect_btc_news():
    """Load Yahoo Finance BTC-USD news cards and retrieve publication times."""
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        try:
            await page.goto(
                NEWS_URL,
                wait_until="domcontentloaded",
                timeout=60000
            )
            await page.wait_for_timeout(5000)

            previous_count = -1
            unchanged_rounds = 0

            # Scroll until no additional cards load for three rounds.
            for round_num in range(1, MAX_SCROLL_ROUNDS + 1):
                current_count = await page.locator(
                    "div.content"
                ).count()

                print(f"Round {round_num}: {current_count} news cards")

                if current_count == previous_count:
                    unchanged_rounds += 1
                else:
                    unchanged_rounds = 0

                if unchanged_rounds >= 3:
                    print("No additional cards loaded for 3 rounds.")
                    break

                previous_count = current_count
                await page.evaluate(
                    "window.scrollTo(0, document.body.scrollHeight)"
                )
                await page.mouse.wheel(0, 3000)
                await page.wait_for_timeout(WAIT_AFTER_SCROLL_MS)

            cards = page.locator("div.content")
            raw_articles = []
            seen_links = set()

            for i in range(await cards.count()):
                card = cards.nth(i)
                title_element = card.locator("h3").first
                link_element = card.locator("a.titles").first
                publisher_element = card.locator("div.publishing").first

                if (
                    await title_element.count() == 0
                    or await link_element.count() == 0
                    or await publisher_element.count() == 0
                ):
                    continue

                title = (await title_element.inner_text()).strip()
                href = await link_element.get_attribute("href")
                link = urljoin(NEWS_URL, href or "")
                publisher_text = (
                    await publisher_element.inner_text()
                ).strip()

                parts = [
                    part.strip()
                    for part in publisher_text.split("•")
                    if part.strip()
                ]

                if not title or not href or not parts or link in seen_links:
                    continue

                seen_links.add(link)
                raw_articles.append({
                    "title": title,
                    "source": parts[0],
                    "link": link
                })

        finally:
            await browser.close()

    print("\nUnique article cards:", len(raw_articles))

    # Retrieve each article's exact publication timestamp.
    collected_articles = []

    for i, article in enumerate(raw_articles, start=1):
        try:
            timestamp = get_published_time(article["link"])
            published_at = parse_timestamp(timestamp)

            collected_articles.append({
                **article,
                "published_at_utc": (
                    published_at.isoformat() if published_at else None
                ),
                "publication_status": (
                    "success" if published_at else "missing_timestamp"
                )
            })

        except requests.RequestException as exc:
            collected_articles.append({
                **article,
                "published_at_utc": None,
                "publication_status": "request_failed",
                "publication_error": str(exc)
            })

        if i % 10 == 0 or i == len(raw_articles):
            print(f"Processed {i}/{len(raw_articles)} articles")

    print("\n========== Collection Summary ==========")
    print("Unique articles:", len(collected_articles))
    print(
        "Exact timestamps retrieved:",
        sum(a["publication_status"] == "success"
            for a in collected_articles)
    )
    print(
        "Missing timestamps:",
        sum(a["publication_status"] == "missing_timestamp"
            for a in collected_articles)
    )
    print(
        "Failed article requests:",
        sum(a["publication_status"] == "request_failed"
            for a in collected_articles)
    )

    return collected_articles


collected_articles = await collect_btc_news()

Round 1: 13 news cards
Round 2: 33 news cards
Round 3: 53 news cards
Round 4: 73 news cards
Round 5: 93 news cards
Round 6: 113 news cards
Round 7: 133 news cards
Round 8: 153 news cards
Round 9: 173 news cards
Round 10: 193 news cards
Round 11: 195 news cards
Round 12: 195 news cards
Round 13: 195 news cards
Round 14: 195 news cards
No additional cards loaded for 3 rounds.

Unique article cards: 194
Processed 10/194 articles
Processed 20/194 articles
Processed 30/194 articles
Processed 40/194 articles
Processed 50/194 articles
Processed 60/194 articles
Processed 70/194 articles
Processed 80/194 articles
Processed 90/194 articles
Processed 100/194 articles
Processed 110/194 articles
Processed 120/194 articles
Processed 130/194 articles
Processed 140/194 articles
Processed 150/194 articles
Processed 160/194 articles
Processed 170/194 articles
Processed 180/194 articles
Processed 190/194 articles
Processed 194/194 articles

========== Collection Summary ==========
Unique articles: 194
Ex

## News Collection Step 2 — Filter the Last 72 Hours
Use one UTC reference time to filter articles by their exact datePublished timestamp. This cell uses the collected in-memory records and does not revisit article pages.

In [ ]:
LOOKBACK_HOURS = 72

collection_time = datetime.now(timezone.utc)
cutoff_time = collection_time - timedelta(hours=LOOKBACK_HOURS)

recent_articles = []

for article in collected_articles:
    published_at = parse_timestamp(article.get("published_at_utc"))

    if (
        published_at is not None
        and cutoff_time <= published_at <= collection_time
    ):
        recent_articles.append(article)

print("Collection time (UTC):", collection_time.isoformat())
print("72-hour cutoff (UTC):", cutoff_time.isoformat())
print("Total collected:", len(collected_articles))
print("Recent 72-hour articles:", len(recent_articles))

if recent_articles:
    timestamps = [
        parse_timestamp(article["published_at_utc"])
        for article in recent_articles
    ]
    print("Newest (UTC):", max(timestamps).isoformat())
    print("Oldest (UTC):", min(timestamps).isoformat())

Collection time (UTC): 2026-10-12T05:28:53.769641+00:00
72-hour cutoff (UTC): 2026-10-09T05:28:53.769641+00:00
Total collected: 194
Recent 72-hour articles: 122
Newest (UTC): 2026-10-12T04:35:19+00:00
Oldest (UTC): 2026-10-09T06:11:53+00:00


## News Collection Step 3 — Extract Summary and Article Content
Enrich only the articles that passed the 72-hour filter. Keep the original headline, source, publication timestamp, and link. If a page does not expose a summary or article body, record that fact rather than generating missing content.

In [ ]:
def extract_article_content(article_url):
    """Extract an available summary and article body from Yahoo Finance."""
    response = session.get(article_url, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    # Summary from page metadata
    summary = None
    for meta in soup.find_all("meta"):
        key = (
            meta.get("property")
            or meta.get("name")
            or ""
        ).lower()

        if key in ["og:description", "description", "twitter:description"]:
            value = (meta.get("content") or "").strip()
            if value:
                summary = value
                break

    # Article body from the page's article-body container
    content = None
    body_element = soup.select_one('[data-testid="article-body"]')

    if body_element:
        paragraphs = [
            p.get_text(" ", strip=True)
            for p in body_element.find_all("p")
            if p.get_text(" ", strip=True)
        ]

        if paragraphs:
            content = "\n\n".join(paragraphs)
        else:
            body_text = body_element.get_text(" ", strip=True)
            if body_text:
                content = body_text

    # JSON-LD fallback for summary or article body
    if not summary or not content:
        def inspect_metadata(data):
            items = data if isinstance(data, list) else [data]

            for item in items:
                if not isinstance(item, dict):
                    continue

                found_summary = item.get("description")
                found_content = item.get("articleBody")

                if (
                    not summary
                    and isinstance(found_summary, str)
                    and found_summary.strip()
                ):
                    return found_summary.strip(), found_content

                if (
                    not content
                    and isinstance(found_content, str)
                    and found_content.strip()
                ):
                    return found_summary, found_content.strip()

                graph = item.get("@graph", [])
                if graph:
                    nested_summary, nested_content = inspect_metadata(graph)
                    if nested_summary or nested_content:
                        return nested_summary, nested_content

            return None, None

        for script in soup.find_all(
            "script",
            type="application/ld+json"
        ):
            try:
                data = json.loads(script.string or script.get_text())
            except (json.JSONDecodeError, TypeError):
                continue

            fallback_summary, fallback_content = inspect_metadata(data)

            if not summary and fallback_summary:
                summary = fallback_summary
            if not content and fallback_content:
                content = fallback_content

            if summary and content:
                if summary.strip() == content.strip():
                    summary = None

    return summary, content


from collections import Counter

for i, article in enumerate(recent_articles, start=1):
    try:
        summary, content = extract_article_content(article["link"])
        article["summary"] = summary
        article["content"] = content

        if summary and content:
            article["content_status"] = "summary_and_content"
        elif summary:
            article["content_status"] = "summary_only"
        elif content:
            article["content_status"] = "content_only"
        else:
            article["content_status"] = "unavailable"

    except requests.RequestException as exc:
        article["summary"] = None
        article["content"] = None
        article["content_status"] = "request_failed"
        article["content_error"] = str(exc)

    if i % 10 == 0 or i == len(recent_articles):
        print(f"Processed {i}/{len(recent_articles)} articles")

print("\n========== Content Extraction Summary ==========")
for status, count in Counter(
    article["content_status"] for article in recent_articles
).items():
    print(f"{status}: {count}")

print("\n========== First 3 Articles ==========")
for i, article in enumerate(recent_articles[:3], start=1):
    print(f"\n{i}. {article['title']}")
    print("Source:", article["source"])
    print("Published (UTC):", article["published_at_utc"])
    print("Summary:", article.get("summary"))
    print("Content preview:", (article.get("content") or "")[:300])
    print("Status:", article["content_status"])

Processed 10/122 articles
Processed 20/122 articles
Processed 30/122 articles
Processed 40/122 articles
Processed 50/122 articles
Processed 60/122 articles
Processed 70/122 articles
Processed 80/122 articles
Processed 90/122 articles
Processed 100/122 articles
Processed 110/122 articles
Processed 120/122 articles
Processed 122/122 articles

========== Content Extraction Summary ==========
summary_and_content: 122

========== First 3 Articles ==========

1. Bitcoin ETF Inflows Stall After $1.7 Billion Burst: Will Demand Return?
Source: BeInCrypto
Published (UTC): 2026-10-12T04:09:36+00:00
Summary: Bitcoin ETF inflows have totaled just $232 million in 13 sessions since a $1.7 billion burst. Can buyers return?
Content preview: US spot Bitcoin (BTC) exchange-traded funds (ETFs) have added only about $232 million over 13 sessions. Bitcoin ETF inflows totaled $1.7 billion across September 21 and 22, then went quiet.

Bitcoin's price recovered to above $83,000 overnight while US-listed funds 

In [ ]:
# Keep the article content and remove duplicate summaries
updated_count = 0

for article in recent_articles:
    summary = article.get("summary")
    content = article.get("content")

    if (
        summary
        and content
        and summary.strip() == content.strip()
    ):
        article["summary"] = None
        article["content_status"] = "content_only"
        updated_count += 1

print("Articles updated:", updated_count)

# Verify the changes
summary_none = sum(
    article.get("summary") is None
    for article in recent_articles
)

remaining_duplicates = sum(
    bool(article.get("summary"))
    and bool(article.get("content"))
    and article["summary"].strip() == article["content"].strip()
    for article in recent_articles
)

print("Total recent articles:", len(recent_articles))
print("Summary is None:", summary_none)
print("Summary still identical to content:", remaining_duplicates)

# Display one updated example
for article in recent_articles:
    if article.get("summary") is None:
        print("\nExample:")
        print("Title:", article["title"])
        print("Summary:", article["summary"])
        print("Content length:", len(article.get("content") or ""))
        print("Status:", article["content_status"])
        break

Articles updated: 7
Total recent articles: 122
Summary is None: 7
Summary still identical to content: 0

Example:
Title: Sector Update: Financial Stocks Rise Friday
Summary: None
Content length: 100
Status: content_only


In [ ]:
from collections import Counter

print("========== Final News Data Quality Check ==========")

print("Total collected:", len(collected_articles))
print("Recent articles:", len(recent_articles))

print("\nMissing required fields:")
for field in [
    "title", "source", "published_at_utc",
    "link", "content"
]:
    missing = sum(
        not article.get(field)
        for article in recent_articles
    )
    print(f"{field}: {missing}")

print(
    "Summary is None:",
    sum(a.get("summary") is None for a in recent_articles)
)

print("\nContent status:")
for status, count in Counter(
    a.get("content_status", "unknown")
    for a in recent_articles
).items():
    print(f"{status}: {count}")

print("\nDuplicate links:", len([
    a.get("link") for a in recent_articles if a.get("link")
]) - len(set(
    a.get("link") for a in recent_articles if a.get("link")
)))

========== Final News Data Quality Check ==========
Total collected: 194
Recent articles: 122

Missing required fields:
title: 0
source: 0
published_at_utc: 0
link: 0
content: 0
Summary is None: 7

Content status:
summary_and_content: 115
content_only: 7

Duplicate links: 0


In [ ]:
# Prepare the recent news dataset
news_df = pd.DataFrame(recent_articles)

# Keep the required fields for news retrieval
columns_to_save = [
    "title",
    "source",
    "published_at_utc",
    "link",
    "summary",
    "content",
    "content_status"
]

news_df = news_df[columns_to_save]

# Create output directory
output_dir = Path("../data/news/raw")
output_dir.mkdir(parents=True, exist_ok=True)

# Save recent 72-hour news
output_path = output_dir / "bitcoin_news_72h.csv"

news_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("News CSV saved successfully.")
print("File:", output_path)
print("Total articles:", len(news_df))
print("Columns:", news_df.columns.tolist())

News CSV saved successfully.
File: ..\data\news\raw\bitcoin_news_72h.csv
Total articles: 122
Columns: ['title', 'source', 'published_at_utc', 'link', 'summary', 'content', 'content_status']
